## Imports

In [0]:
from pyspark.sql.functions import (
    col,
    lower,
    upper,
    trim,
    to_date,
    current_timestamp,
    row_number,
    lit,
    when,
    lpad,
    current_date
)
from pyspark.sql.window import Window

## Bronze table

In [0]:
bronze_df = spark.table("bronze_loyalty")
silver_customers = spark.table("silver_customers")

display(bronze_df.limit(10))
print(f"Bronze rows: {bronze_df.count():,}")

loyalty_account_id,customer_id,join_date,tier,points_balance,loyalty_status,created_at,updated_at,source_system
LOY000000001,CRM0093900,2024-06-26T16:23:16.278Z,SILVER,161365,ACTIVE,2024-06-26T16:23:16.278Z,2024-07-08T16:23:16.278Z,loyalty_platform
LOY000000002,CRM0066969,2027-02-18T16:23:16.278Z,SILVER,112118,ACTIVE,2027-02-18T16:23:16.278Z,2027-08-20T16:23:16.278Z,loyalty_platform
LOY000000003,CRM0082818,2023-12-24T16:23:16.278Z,BRONZE,146221,ACTIVE,2023-12-24T16:23:16.278Z,2024-01-11T16:23:16.278Z,loyalty_platform
LOY000000004,CRM0023557,2025-10-02T16:23:16.278Z,BRONZE,170081,ACTIVE,2025-10-02T16:23:16.278Z,2026-09-24T16:23:16.278Z,loyalty_platform
LOY000000005,CRM0061886,2021-09-19T16:23:16.278Z,SILVER,228681,ACTIVE,2021-09-19T16:23:16.278Z,2022-01-19T16:23:16.278Z,loyalty_platform
LOY000000006,CRM0024863,2021-12-02T16:23:16.278Z,BRONZE,204384,ACTIVE,2021-12-02T16:23:16.278Z,2022-08-24T16:23:16.278Z,loyalty_platform
LOY000000007,CRM0004455,2021-12-22T16:23:16.278Z,BRONZE,158209,ACTIVE,2021-12-22T16:23:16.278Z,2022-08-06T16:23:16.278Z,loyalty_platform
LOY000000008,CRM0054245,2026-05-16T16:23:16.278Z,SILVER,213032,ACTIVE,2026-05-16T16:23:16.278Z,2026-12-25T16:23:16.278Z,loyalty_platform
LOY000000009,CRM0029388,2024-05-29T16:23:16.278Z,BRONZE,176713,ACTIVE,2024-05-29T16:23:16.278Z,2025-04-20T16:23:16.278Z,loyalty_platform
LOY000000010,CRM0090134,2021-12-24T16:23:16.278Z,GOLD,227230,ACTIVE,2021-12-24T16:23:16.278Z,2022-04-04T16:23:16.278Z,loyalty_platform


Bronze rows: 55,550


## Silver transformation

In [0]:
VALID_TIERS = ["BRONZE", "SILVER", "GOLD", "PLATINUM"]
VALID_STATUSES = ["ACTIVE", "INACTIVE", "SUSPENDED"]

standardised_df = (
    bronze_df
    .withColumn("join_date", to_date(col("join_date")))
    .withColumn("tier", upper(trim(col("tier"))))
    .withColumn("loyalty_status", upper(trim(col("loyalty_status"))))
    .withColumn("created_at", to_date(col("created_at")))
    .withColumn("updated_at", to_date(col("updated_at")))
    .withColumn("_silver_processed_at", current_timestamp())
)
window_spec = Window.partitionBy("loyalty_account_id").orderBy(
    col("updated_at").desc_nulls_last()
)
deduped_df = (
    standardised_df
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)
customer_lookup = (
    silver_customers
    .select("customer_id")
    .dropDuplicates(["customer_id"])
    .withColumn("customer_exists", lit(True))
)
loyalty_with_customer_check = (
    deduped_df
    .join(
        customer_lookup,
        on="customer_id",
        how="left"
    )
)
quality_df = (
    loyalty_with_customer_check
    .withColumn(
        "_has_future_date",
        (
            (col("join_date") > current_date()) |
            (col("updated_at") > current_date())
        )
    )
    .withColumn(
        "_is_valid",
        when(col("loyalty_account_id").isNull(), lit(False))
        .when(col("customer_id").isNull(), lit(False))
        .when(col("customer_exists").isNull(), lit(False))
        .when(col("join_date").isNull(), lit(False))
        .when(col("tier").isNull(), lit(False))
        .when(~col("tier").isin(VALID_TIERS), lit(False))
        .when(col("loyalty_status").isNull(), lit(False))
        .when(~col("loyalty_status").isin(VALID_STATUSES), lit(False))
        .when(col("points_balance").isNull(), lit(False))
        .when(col("points_balance") < 0, lit(False))
        .otherwise(lit(True))
    )
    .withColumn(
        "_rejection_reason",
        when(col("loyalty_account_id").isNull(), lit("loyalty_account_id is null"))
        .when(col("customer_id").isNull(), lit("customer_id is null"))
        .when(col("customer_exists").isNull(), lit("customer_id does not exist in silver customers"))
        .when(col("join_date").isNull(), lit("join_date is null"))
        .when(col("tier").isNull(), lit("tier is null"))
        .when(~col("tier").isin(VALID_TIERS), lit("invalid tier"))
        .when(col("loyalty_status").isNull(), lit("loyalty_status is null"))
        .when(~col("loyalty_status").isin(VALID_STATUSES), lit("invalid loyalty_status"))
        .when(col("points_balance").isNull(), lit("points_balance is null"))
        .when(col("points_balance") < 0, lit("points_balance must be greater than or equalto zero"))
        .otherwise(lit(None))
    )
)
valid_loyalty = quality_df.filter(col("_is_valid") == True)
rejected_loyalty = quality_df.filter(col("_is_valid") == False)

## Delta tables

In [0]:
(
    valid_loyalty
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_loyalty")
)

(
    rejected_loyalty
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("rejected_loyalty")
)

print("Silver Loyalty completed")
print(f"Bronze rows: {bronze_df.count():,}")
print(f"After deduplication: {deduped_df.count():,}")
print(f"Valid loyalty: {valid_loyalty.count():,}")
print(f"Rejected loyalty: {rejected_loyalty.count():,}")

Silver Loyalty completed
Bronze rows: 55,550
After deduplication: 55,000
Valid loyalty: 53,720
Rejected loyalty: 1,280


In [0]:
spark.sql("SHOW TABLES").show()

+--------+------------------+-----------+
|database|         tableName|isTemporary|
+--------+------------------+-----------+
| default|  bronze_addresses|      false|
| default|  bronze_customers|      false|
| default|    bronze_loyalty|      false|
| default|     bronze_orders|      false|
| default|   bronze_payments|      false|
| default|rejected_addresses|      false|
| default|rejected_customers|      false|
| default|  rejected_loyalty|      false|
| default|   rejected_orders|      false|
| default| rejected_payments|      false|
| default|  silver_addresses|      false|
| default|  silver_customers|      false|
| default|    silver_loyalty|      false|
| default|     silver_orders|      false|
| default|   silver_payments|      false|
+--------+------------------+-----------+

